# Technical Diary — Sections 2, 3, 4

Synthetic genomic dataset generation using GIAB HG005 data.

- Section 2: Data Source & FASTQ format verification
- Section 3: Creating Synthetic Samples by Splitting
- Section 4: Read Count Validation

## Imports

In [ ]:
import os

## 2. Data Source

**Genome in a Bottle (GIAB)**

- Sample used: HG005
- Download: https://zenodo.org/records/7310196
- Data type: Whole Genome Sequencing (WGS), Paired-end reads (R1 & R2)

Paired-end reads:
- `HG005_Son.R1.fastq.gz`
- `HG005_Son.R2.fastq.gz`

### Verifying FASTQ format

In [ ]:
r1_path = "HG005_Son.R1.10X.fastq"
r2_path = "HG005_Son.R2.10X.fastq"

def peek_first_read(file_path, label):
    print(f"--- First read of {label} ---")
    with open(file_path) as f:
        for _ in range(4):
            print(f.readline().strip())
    print("\n")

peek_first_read(r1_path, "R1")
peek_first_read(r2_path, "R2")

## 3. Creating Synthetic Samples by Splitting

Splitting into 5 samples paired reads as 10 files.

In [ ]:
r1_path = "HG005_Son.R1.10X.fastq"
r2_path = "HG005_Son.R2.10X.fastq"

os.makedirs("results", exist_ok=True)

def fastq_generator(file_path):
    """Yield one read (4 lines) at a time"""
    with open(file_path) as f:
        while True:
            record = [f.readline().strip() for _ in range(4)]
            if not record[0]:
                break
            yield record

r1_gen = fastq_generator(r1_path)
r2_gen = fastq_generator(r2_path)

# samples and reads per sample
samples = ["sample1", "sample2", "sample3", "sample4", "sample5"]
reads_per_sample = 1000000

# Splitting
for sample in samples:
    r1_out = f"results/{sample}_R1.fastq"
    r2_out = f"results/{sample}_R2.fastq"
    with open(r1_out, "w") as o1, open(r2_out, "w") as o2:
        for _ in range(reads_per_sample):
            r1_record = next(r1_gen)
            r2_record = next(r2_gen)
            o1.write("\n".join(r1_record) + "\n")
            o2.write("\n".join(r2_record) + "\n")
    print(f"{sample} created: {r1_out} + {r2_out}")

print("FASTQ splitting complete!")

In [ ]:
# Results
files = os.listdir("results")
print("Files in results folder:", files)

with open("results/sample1_R1.fastq") as f:
    print("\nFirst read in sample1_R1.fastq:")
    for _ in range(4):
        print(f.readline().strip())  # this is optional

## 4. Read Count Validation

Paired-end integrity confirmed if the read count is 100k for each sample. It should have the same read counts for each paired-end sample.

In [ ]:
def count_reads(file):
    return sum(1 for _ in open(file)) // 4

for f in os.listdir("results"):
    print(f"{f}: {count_reads(os.path.join('results', f))} reads")